# 04. SA Benders 실행

확인 항목:
1. SA Benders 실행
2. energy 순 sample 검사
3. MP-feasible sample 선택
4. continuous / encoded MP Gurobi 기준과 비교
5. 실패 상태 보존

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from cflp_bd_qa_ex2.config.loader import load_config
from cflp_bd_qa_ex2.data.io import load_all_instances

cfg = load_config(ROOT / "config" / "experiments" / "pilot.yaml")
instances = load_all_instances(ROOT / "data" / "raw")
by_id = {i.instance_id: i for i in instances}
print("configuration_hash:", cfg["configuration_hash"][:16])


configuration_hash: 0dc9f18b09b09302


## 1. SA end-to-end Benders

SA 가 만든 candidate `y` 로 SP 를 풀고 cut 을 추가한다.
`gurobi_controlled` 와 cut trajectory 가 달라질 수 있으므로 **합치지 않는다.**

certified lower bound 는 SA 가 아니라 **매 iteration Gurobi continuous MP** 에서 온다.

In [2]:
from cflp_bd_qa_ex2.benders.solver import QuboMasterProvider, run_benders
from cflp_bd_qa_ex2.solvers.sa import run_sa
from cflp_bd_qa_ex2.status import TrajectorySource

target_ids = ["4x12_s100", "8x25_s200", "16x50_s300"]
traces, qubo_records = {}, {}
for tid in target_ids:
    if tid not in by_id:
        continue
    collected = []
    provider = QuboMasterProvider(TrajectorySource.SA_END_TO_END, run_sa, collected)
    traces[tid] = run_benders(by_id[tid], provider, cfg)
    qubo_records[tid] = collected

runs = pd.DataFrame([t.run.as_record() for t in traces.values()])
runs[["instance_id", "termination_status", "benders_iterations",
      "certified_lower_bound", "best_upper_bound", "final_optimality_gap",
      "dual_minimality_status", "dual_guard_action"]]


,instance_id,termination_status,benders_iterations,certified_lower_bound,best_upper_bound,final_optimality_gap,dual_minimality_status,dual_guard_action
0,4x12_s100,STALLED_DUPLICATE_CUT,3,4815.644243,4904.433098,0.018104,MINIMAL,NONE
1,8x25_s200,STALLED_DUPLICATE_CUT,6,9221.364484,9596.821799,0.039123,MINIMAL,NONE
2,16x50_s300,STALLED_DUPLICATE_CUT,57,16651.987028,17459.428179,0.046247,MINIMAL,NONE


## 2. Energy 순 sample 검사와 MP-feasible 선택

선택 규칙:
1. energy 오름차순 정렬
2. `y`, `theta` 를 original scale 로 decode
3. capacity constraint 와 **모든 활성 cut** 검사
4. MP-feasible 한 **첫** sample 선택
5. 없으면 `NO_FEASIBLE_SAMPLE`

feasibility 는 slack equality residual 이 아니라 **원래 MP 부등식**으로 판정한다.

In [3]:
from cflp_bd_qa_ex2.qubo.builder import build_master_qubo
from cflp_bd_qa_ex2.qubo.decode import select_feasible_sample

tid = target_ids[0]
qubo = build_master_qubo(by_id[tid], [], cfg)
output = run_sa(qubo, cfg)

energies = [energy for _, energy, _, _ in output["samples"]]
assert energies == sorted(energies), "energy 오름차순이 아니다"

selected, decoded, status = select_feasible_sample(qubo, by_id[tid], [], output["samples"], cfg)
inspection = pd.DataFrame([d.as_record() for d in decoded[:12]])
print("status:", status, "| 선택된 rank:", selected.sample_rank if selected else None)
inspection[["sample_rank", "energy", "decoded_y", "decoded_theta",
            "capacity_violation", "max_cut_violation", "mp_feasible",
            "penalty_residual", "num_occurrences"]]


status: SUCCESS | 선택된 rank: 0


,sample_rank,energy,decoded_y,decoded_theta,capacity_violation,max_cut_violation,mp_feasible,penalty_residual,num_occurrences
0,0,3223.213314,"[1, 0, 0, 1]",0.000000,0.0,0.0,True,0.0,100
1,1,3236.055488,"[1, 0, 0, 1]",12.842174,0.0,0.0,True,0.0,2
2,2,3661.235655,"[0, 1, 1, 1]",0.000000,0.0,0.0,True,0.0,123
3,3,3674.077829,"[0, 1, 1, 1]",12.842174,0.0,0.0,True,0.0,5
4,4,3970.798184,"[1, 0, 1, 1]",0.000000,0.0,0.0,True,0.0,177
5,5,3983.640357,"[1, 0, 1, 1]",12.842174,0.0,0.0,True,0.0,2
6,6,4437.556402,"[1, 1, 0, 1]",0.000000,0.0,0.0,True,0.0,110
7,7,5185.141271,"[1, 1, 1, 1]",0.000000,0.0,0.0,True,0.0,134
8,8,5197.983445,"[1, 1, 1, 1]",12.842174,0.0,0.0,True,0.0,1
9,9,17674.511530,"[1, 0, 0, 1]",0.000000,0.0,0.0,True,1.0,78


`penalty_residual` 이 0 이 아닌데도 `mp_feasible` 이 True 인 행이 있는지 확인한다.

**두 가지 서로 다른 이유가 겹친다.**

1. **feasibility 판정 기준이 다르다.** MP feasibility 는 원래 부등식
   (`sum_j s_j y_j >= D`, `theta + sum_j s_j v_j y_j >= sum_i u_i`) 으로만 본다.
   QUBO 의 equality residual 은 slack 변수가 잉여를 **정확히** 표현할 때만 0 이 된다.
   따라서 cut 이 하나도 없어도, capacity slack bit 가 실제 잉여와 다른 값을 가지면
   residual 은 0 이 아니면서 MP-feasible 일 수 있다.
2. **cut slack 은 애초에 정확히 표현되지 않는다.** capacity slack 은 `s_j`, `D` 가
   정수라 `delta = 1` 로 정확히 표현되지만, cut slack 은 dual `u`, `v` 가 실수라
   격자에 맞지 않는다.

그러므로 **MP-feasible sample 이 penalty residual 0 을 갖는다고 가정하면 안 된다.**
반대로 residual 이 크다고 해서 infeasible 한 것도 아니다.


In [4]:
from cflp_bd_qa_ex2.qubo.decode import feasible_sample_rate

feasible = [d for d in decoded if d.mp_feasible]
residual_positive = sum(1 for d in feasible if d.penalty_residual > 1e-9)

print(f"MP-feasible sample 종류: {len(feasible)} / {len(decoded)}")
if feasible:
    print(f"그중 penalty_residual > 1e-9 인 비율: "
          f"{residual_positive / len(feasible):.3f}")
    print(f"penalty_residual 최대: {max(d.penalty_residual for d in feasible):.6g}")
else:
    print("MP-feasible sample 없음")

# occurrence 가중 비율은 공용 함수를 쓴다 (notebook 에서 재구현하지 않는다).
print(f"MP-feasible sample rate (occurrence 가중): {feasible_sample_rate(decoded):.4f}")


MP-feasible sample 종류: 20 / 21
그중 penalty_residual > 1e-9 인 비율: 0.550
penalty_residual 최대: 41
MP-feasible sample rate (occurrence 가중): 0.9820


## 3. Gurobi 기준과 비교

SA 의 decoded MP objective 를 continuous MP / encoded MP 최적값과 비교한다.
**SA 목적값은 lower bound 로 쓰지 않는다.**

In [5]:
from cflp_bd_qa_ex2.encoding.theta import build_theta_encoding
from cflp_bd_qa_ex2.models.master import solve_continuous_mp, solve_encoded_mp

rows = []
for tid, trace in traces.items():
    inst = by_id[tid]
    theta_encoding = build_theta_encoding(inst.objective_upper_bound,
                                          int(cfg["encoding"]["theta"]["bits"]))
    for it in trace.iterations:
        fields = it.as_record()
        rows.append({
            "instance_id": tid,
            "benders_iteration": fields["benders_iteration"],
            "sa_energy": fields.get("energy"),
            "sa_decoded_mp_objective": fields.get("decoded_mp_objective"),
            "continuous_mp_objective": fields.get("continuous_mp_objective"),
            "certified_lower_bound": fields.get("certified_lower_bound"),
            "candidate_original_objective": fields.get("candidate_original_objective"),
            "mp_feasible": fields.get("mp_feasible"),
            "mp_feasible_sample_rate": fields.get("mp_feasible_sample_rate"),
        })

comparison = pd.DataFrame(rows)
comparison


,instance_id,benders_iteration,sa_energy,sa_decoded_mp_objective,continuous_mp_objective,certified_lower_bound,candidate_original_objective,mp_feasible,mp_feasible_sample_rate
0,4x12_s100,1,3.223213e+03,3223.213313,3223.213313,3223.213313,4904.433098,True,0.982
1,4x12_s100,2,6.275617e+03,6032.724744,4815.644243,4815.644243,6180.326246,True,0.820
2,4x12_s100,3,6.671738e+03,6328.094742,4815.644243,4815.644243,6180.326246,True,0.768
3,8x25_s200,1,6.911147e+03,6911.147382,6911.147382,6911.147382,9596.821799,True,0.973
4,8x25_s200,2,1.162784e+04,11599.371991,8898.287135,8898.287135,9804.916630,True,0.735
...,...,...,...,...,...,...,...,...,...
61,16x50_s300,53,1.265094e+11,105065.847028,16651.987028,16651.987028,17723.542561,True,0.429
62,16x50_s300,54,1.099170e+11,105489.208209,16651.987028,16651.987028,18169.272072,True,0.442
63,16x50_s300,55,1.171640e+11,105286.699852,16651.987028,16651.987028,17990.628879,True,0.450
64,16x50_s300,56,9.037132e+10,107288.813369,16651.987028,16651.987028,20418.283130,True,0.443


## 4. 실패 상태 보존

실패를 성공으로 바꾸거나 행을 삭제하지 않는다.

In [6]:
iteration_records = []
for trace in traces.values():
    iteration_records.extend(it.as_record() for it in trace.iterations)

frame = pd.DataFrame(iteration_records)
print("candidate_status 분포:")
print(frame["candidate_status"].value_counts())
print()
print("termination_status 분포:")
print(runs["termination_status"].value_counts())


candidate_status 분포:
candidate_status
SUCCESS    66
Name: count, dtype: int64

termination_status 분포:
termination_status
STALLED_DUPLICATE_CUT    3
Name: count, dtype: int64


## 5. 결과 저장

In [7]:
from cflp_bd_qa_ex2.evaluation.results import attach_provenance, save_records

formats = cfg["output"]["formats"]
save_records(attach_provenance(iteration_records, cfg), ROOT / "results" / "sa",
             "sa_iterations", formats,
             merge_keys=("instance_id", "trajectory_source", "benders_iteration"))
save_records(attach_provenance(runs.to_dict("records"), cfg), ROOT / "results" / "sa",
             "sa_runs", formats, merge_keys=("instance_id", "trajectory_source"))
print("저장 완료")


저장 완료


## 해석

- SA 가 작은 QUBO 에서 ground state 를 찾더라도, 그것이 Benders 수렴을 보장하지 않는다.
  QUBO ground state 는 penalized 문제의 해이고, Benders 수렴은 MP 최적성이 필요하다.
- `mp_feasible_sample_rate` 가 iteration 과 함께 떨어지면 cut 이 늘면서 QUBO 가
  어려워지고 있다는 신호이다.
- `certified_lower_bound` 는 항상 Gurobi continuous MP 에서 온다.
  SA energy 와 혼동하면 안 된다.